# ByteLLM - benchmarks sur Google Colab

Active un GPU: *Execution > Modifier le type d'execution > GPU (T4)*.

Ce notebook: (1) installe le depot, (2) lance les tests, (3) mesure le **BPB sur enwik8** face a zlib/bz2/lzma, (4) fait l'**ablation du raisonnement recurrent**, (5) mesure la **validite des fichiers generes** avec/sans contraintes de grammaire.

> Les chiffres d'un petit entrainement ne sont pas ceux d'un modele pret a l'emploi: ils servent a comparer des configurations.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
!rm -rf bytellm_repo && git clone https://github.com/cheikhoudieng/bytellm bytellm_repo
%cd bytellm_repo
!pip install -q -e .

## 1. Tests unitaires (causalite, codecs binaires, corpus)

In [ ]:
!python tests/test_bytellm.py

## 2. Smoke test (~2 min): verifie que tout tourne

In [ ]:
!python -m bytellm.benchmark --preset tiny --steps 200 --max-mb 2 --batch-size 16 --amp --out-dir bench_out/smoke

## 3. Benchmark enwik8 (preset `small`, ~20M parametres)
Augmente `--steps` et `--max-mb` (jusqu'a 100) si le temps le permet. Reference: un modele d'octets bien entraine vise ~1.1-1.3 BPB sur enwik8; zlib ~2.9.

In [ ]:
!python -m bytellm.benchmark --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 3 --out-dir bench_out/main

## 4. Ablation: le raisonnement recurrent aide-t-il ?
Meme configuration, `--reasoning-steps 1` (pas de pondering). Si le BPB n'est pas meilleur avec 3 etapes, le coeur de raisonnement n'apporte rien a cette echelle.

In [ ]:
!python -m bytellm.benchmark --preset small --steps 3000 --max-mb 20 --batch-size 32 --seq-len 256 --amp --reasoning-steps 1 --domains 0 --out-dir bench_out/main

## 5. Resultats

In [ ]:
import json, glob
import matplotlib.pyplot as plt
rows = []
for f in sorted(glob.glob('bench_out/main/results_*.json')):
    r = json.load(open(f))
    rows.append((f"{r['preset']} r={r['reasoning_steps']}", r['test_bpb']))
    print(f"{r['preset']} reasoning_steps={r['reasoning_steps']}: params={r['parameters']:,} test_bpb={r['test_bpb']} "
          f"({r['speed']['forward_bytes_per_sec']:.0f} B/s fwd, train {r['train_seconds']}s)")
base = r['baselines_bpb']; print('baselines (bpb):', base)
labels = [n for n, _ in rows] + list(base)
vals = [v for _, v in rows] + list(base.values())
plt.figure(figsize=(8, 3.5)); plt.barh(labels, vals); plt.xlabel('bits per byte (lower is better)'); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
r = json.load(open(sorted(glob.glob('bench_out/main/results_small_r3.json'))[0]))
print('BPB synthetique held-out:', r['domains']['synthetic_heldout_bpb'])
print(f"{'domaine':12} {'contraint':>10} {'sans contrainte':>16}")
for d, v in r['domains']['validity_rate'].items():
    print(f"{d:12} {v['constrained']:>10.2f} {v['unconstrained']:>16.2f}")

## 6. Sauvegarder les resultats et checkpoints dans Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bytellm && cp -r bench_out /content/drive/MyDrive/bytellm/